# Notebook 22 — MegaPose + ViSP: CAD-Based 6D Pose & Tracking

---

**Part 7 · Deep Learning 6D Pose**

```
┌─────────────────────────────────────────────────────────────────────┐
│   MegaPose Pipeline                                                 │
│                                                                     │
│   CAD Model (OBJ) ──► Render views ──► MegaPose ──► Initial Pose   │
│   RGB-D Frame     ──► Detection    ──┘             │               │
│                                                    ▼               │
│                                              ViSP Tracker          │
│                                              (frame-to-frame)      │
└─────────────────────────────────────────────────────────────────────┘
```

> **MegaPose** gives you an initial 6D pose estimate from a CAD model alone — no training on your specific object required. **ViSP** then tracks that pose robustly across video frames.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install numpy matplotlib opencv-python -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
print('Core imports OK')

## Learning Objectives

By the end of this notebook you will:

1. Understand **MegaPose** — how it uses CAD models + render-and-compare for initial pose  
2. Know the exact **3-file CAD model format** required (`model.obj`, `model.mtl`, `texture.jpg`)  
3. Understand how **ViSP** extends MegaPose into real-time pose *tracking*  
4. Know the full MegaPose + ViSP setup procedure  
5. Simulate the estimation → tracking pipeline in Python  
6. Compare MegaPose, FoundationPose, and FreeZe side-by-side

## 1 — What Is MegaPose?

**MegaPose** (Labbe et al., 2022, INRIA) is a 6D pose estimator that:

- Takes a **3D CAD model** of your object
- Takes an **RGB image** of the scene with a detected 2D bounding box
- Returns a 6D pose **without any object-specific training**

### Core Idea: Render-and-Compare

```
Step 1: Render the CAD model from many viewpoints
  ┌────────────────────────────────────────────────┐
  │  Render 1   Render 2   Render 3   Render N     │
  │  (0°,0°)   (30°,0°)  (60°,0°)  (all angles)  │
  └────────────────────────────────────────────────┘

Step 2: Extract features from renders AND scene crop
  [Feature encoder trained on millions of CAD models]

Step 3: Compare scene features to rendered features → best match
  scene_crop_features ≈ render_features at (30°, 15°, Z=0.4m)

Step 4: Refine pose with iterative optimization
  → Final 6D pose [R | t]
```

### Why It Generalizes

MegaPose was trained on **2 million+ diverse 3D objects** (ShapeNet + Google Scanned Objects + Objaverse). The feature encoder learned to match **geometry** regardless of texture or object category. This means:

> If your object has a CAD model, MegaPose can estimate its pose. No retraining needed.

## 2 — CAD Model Format Requirements

MegaPose (and ViSP) require a specific 3-file folder structure:

```
my_object/
├── model.obj       ← 3D mesh geometry
├── model.mtl       ← Material definition (links to texture)
└── texture.jpg     ← Texture image (can be photo of object)
```

### File Descriptions

**`model.obj`** — Standard 3D mesh format:
```
# OBJ file
mtllib model.mtl
v 0.1 0.0 0.0      # vertex positions
v 0.0 0.1 0.0
...
vt 0.5 0.0          # texture coordinates (UV)
...
vn 0 0 1            # vertex normals
...
usemtl Material
f 1/1/1 2/2/2 3/3/3  # faces (vertex/texture/normal)
```

**`model.mtl`** — Material file:
```
newmtl Material
Ka 1.000000 1.000000 1.000000
Kd 1.000000 1.000000 1.000000
map_Kd texture.jpg    # link to texture
```

**`texture.jpg`** — Can be:
- A photo of the real object (best)
- A downloaded texture that matches the material (e.g., wood from Blender Kit)
- A solid color image for textureless objects

### How to Create the CAD Model

| Tool | Use Case | Difficulty |
|------|----------|------------|
| **Blender** | Modeling from scratch, exporting OBJ | Medium |
| **FreeCAD** | Engineering parts (from technical drawings) | Medium |
| **Meshlab** | Cleaning scanned meshes | Easy |
| **3D Scanner** | Scan a real object directly | Easy (hardware needed) |
| **Online libraries** | Grab off Thingiverse/GrabCAD | Very easy |

In [ ]:
# Create a minimal valid OBJ model (a simple cube) to demonstrate the format
# In a real workflow you'd export this from Blender.

def create_box_obj(size=0.05):
    """Create a minimal box OBJ as a string. Size in meters."""
    s = size / 2
    obj_content = f"""# Minimal box model for MegaPose
# Units: meters, size = {size}m x {size}m x {size}m
mtllib model.mtl

# 8 vertices of a box
v -{s} -{s} -{s}
v  {s} -{s} -{s}
v  {s}  {s} -{s}
v -{s}  {s} -{s}
v -{s} -{s}  {s}
v  {s} -{s}  {s}
v  {s}  {s}  {s}
v -{s}  {s}  {s}

# UV coordinates
vt 0.0 0.0
vt 1.0 0.0
vt 1.0 1.0
vt 0.0 1.0

# Normals (one per face)
vn  0  0 -1
vn  0  0  1
vn -1  0  0
vn  1  0  0
vn  0 -1  0
vn  0  1  0

usemtl Material

# 6 faces (2 triangles each)
f 1/1/1 2/2/1 3/3/1
f 1/1/1 3/3/1 4/4/1
f 5/1/2 7/3/2 6/2/2
f 5/1/2 8/4/2 7/3/2
f 1/1/3 5/2/3 8/3/3
f 1/1/3 8/3/3 4/4/3
f 2/1/4 6/2/4 7/3/4
f 2/1/4 7/3/4 3/4/4
f 1/1/5 2/2/5 6/3/5
f 1/1/5 6/3/5 5/4/5
f 4/1/6 7/3/6 3/2/6
f 4/1/6 8/4/6 7/3/6
"""
    return obj_content

def create_mtl():
    return """newmtl Material
Ka 1.000000 1.000000 1.000000
Kd 1.000000 1.000000 1.000000
Ks 0.000000 0.000000 0.000000
illum 1
map_Kd texture.jpg
"""

# Print the OBJ content
print('=== model.obj (first 20 lines) ===')
obj_str = create_box_obj(size=0.08)
for i, line in enumerate(obj_str.split('\n')[:20]):
    print(f'  {line}')
print('  ...')

print('\n=== model.mtl ===')
for line in create_mtl().split('\n'):
    print(f'  {line}')

# Save to disk (optional — useful if you want to verify with a 3D viewer)
import os
model_dir = '../assets/sample_model'
os.makedirs(model_dir, exist_ok=True)
with open(f'{model_dir}/model.obj', 'w') as f:
    f.write(create_box_obj(size=0.08))
with open(f'{model_dir}/model.mtl', 'w') as f:
    f.write(create_mtl())

# Create a simple solid-color texture
texture = np.ones((256, 256, 3), dtype=np.uint8) * np.array([180, 140, 100], dtype=np.uint8)
cv2.imwrite(f'{model_dir}/texture.jpg', texture)

print(f'\nModel files saved to: {model_dir}/')
print('  model.obj')
print('  model.mtl')
print('  texture.jpg')

## 3 — MegaPose Inference Pipeline

MegaPose consists of two neural networks:

### Network 1: Coarse Estimator

```
Input:  Cropped scene image + rendered model images (many viewpoints)
Output: Initial pose hypothesis (coarse)

Process:
  scene_crop ──► Encoder ──► scene_feat
  render_i   ──► Encoder ──► render_feat_i

  similarity = scene_feat · render_feat_i  (dot product)
  best_i = argmax(similarity)
  coarse_pose = pose_at_viewpoint_i
```

### Network 2: Refiner

```
Input:  scene_crop + rendered image at coarse_pose
Output: Δ[R | t] correction

Process (iterative):
  for N iterations:
    render model at current_pose
    compare render to scene (feature difference)
    predict correction Δ[R | t]
    current_pose = current_pose ⊕ Δ[R | t]

Final: refined_pose
```

### Key Advantage

Both networks are trained on **millions of diverse objects**, so they learn a **generalizable** render-vs-scene comparison. They never need to see YOUR specific object.

This is fundamentally different from EfficientPose, which is trained on specific LineMOD objects.

## 4 — ViSP: Visual Servoing Platform (Tracker)

**ViSP** (INRIA Rennes) is an open-source library for visual servoing and pose estimation, written in C++ with Python bindings.

### Why MegaPose Alone Isn't Enough for Video

MegaPose is an **estimator** — it takes a single frame and returns a pose. For video:

```
Running MegaPose on every frame:
  Frame 1: estimate from scratch → 500ms
  Frame 2: estimate from scratch → 500ms
  Frame 3: estimate from scratch → 500ms
  → Only 2 FPS. Not real-time.
```

### ViSP as a Tracker

```
Frame 1: MegaPose estimates initial pose       → 500ms (once)
Frame 2: ViSP tracks from frame 1 → frame 2   → 5ms
Frame 3: ViSP tracks from frame 2 → frame 3   → 5ms
...
→ ~200 FPS tracking after initialization!
```

### How ViSP Tracking Works

ViSP uses **model-based tracking** with edges and texture:

```
Given: 3D model + previous pose
Step 1: Project 3D model edges onto image using previous pose
Step 2: Search for real edges near projected lines (1D search)
Step 3: Compute residuals (predicted vs actual edge positions)
Step 4: Update pose to minimize residuals (VVS: Virtual Visual Servoing)
Step 5: Repeat until converged
→ New pose
```

This is purely **classical** — no neural network during tracking.

### Combined Architecture

```
                MegaPose (once)
                      │
                      ▼ initial pose
              ┌───────────────────┐
              │                   │
  Frame t ───►│  ViSP Tracker    │──► Pose at frame t
              │  (5ms/frame)      │
              └───────────────────┘
                      │
                      ▼ if tracking lost
                  Re-initialize with MegaPose
```

In [ ]:
# Simulate the full MegaPose + ViSP pipeline

def euler_to_R(rx, ry, rz):
    cx, sx = np.cos(rx), np.sin(rx)
    cy, sy = np.cos(ry), np.sin(ry)
    cz, sz = np.cos(rz), np.sin(rz)
    Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return Rz @ Ry @ Rx

def simulate_megapose_init(frame_rgb, obj_bbox):
    """
    Simulate MegaPose initial pose estimation.
    In reality: loads MegaPose server, renders CAD views, runs inference.
    """
    print('[MegaPose] Loading CAD model...')
    print('[MegaPose] Rendering 72 viewpoints...')
    print('[MegaPose] Running coarse estimator...')
    print('[MegaPose] Running 5 refiner iterations...')
    # Return mock initial pose
    R_init = euler_to_R(0.3, 0.5, 0.1)
    t_init = np.array([0.05, -0.03, 0.45])
    print(f'[MegaPose] Initial pose estimated. t={t_init.round(3)}')
    return R_init, t_init

def simulate_visp_track(R_prev, t_prev, frame_idx):
    """
    Simulate ViSP frame-to-frame tracking.
    In reality: projects 3D edges, matches to image edges, minimizes residuals.
    """
    # Simulate slight object motion per frame
    delta_t = np.array([0.002, 0.001, -0.001])  # ~2mm per frame
    delta_r = np.array([0.01, 0.005, 0.0])       # ~0.5 deg per frame
    R_new = euler_to_R(*delta_r) @ R_prev
    t_new = t_prev + delta_t
    return R_new, t_new

# ── Simulate pipeline ──
print('=' * 55)
print('  MegaPose + ViSP Tracking Pipeline Simulation')
print('=' * 55)

# Mock frame data
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)
obj_bbox = (220, 180, 200, 180)  # x, y, w, h

# Step 1: MegaPose initialization (slow, once)
print('\n[Frame 0] MegaPose initialization...')
R, t = simulate_megapose_init(dummy_frame, obj_bbox)

# Step 2: ViSP tracking (fast, every frame)
print('\n[ViSP Tracking]')
trajectory = [t.copy()]
for frame_idx in range(1, 20):
    R, t = simulate_visp_track(R, t, frame_idx)
    trajectory.append(t.copy())
    if frame_idx % 5 == 0:
        print(f'  Frame {frame_idx:2d}: t = [{t[0]:.4f}, {t[1]:.4f}, {t[2]:.4f}] m')

trajectory = np.array(trajectory)

# Plot trajectory
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ['X (m)', 'Y (m)', 'Z (m)']
colors = ['#e74c3c', '#2ecc71', '#3498db']
for i, (ax, label, color) in enumerate(zip(axes, labels, colors)):
    ax.plot(trajectory[:, i], 'o-', color=color, lw=2, ms=4)
    ax.set_xlabel('Frame', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    ax.set_title(f'Object {label} over Time', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('MegaPose + ViSP: Tracked Object Trajectory', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5 — Full Setup Instructions

### Step 1: Install ViSP

```bash
# Option A: pip (Python bindings only)
pip install visp

# Option B: from source (full features)
# https://visp-doc.inria.fr/doxygen/visp-daily/tutorial-install-ubuntu.html
sudo apt install libvisp-dev python3-visp

# Option C: conda
conda install -c conda-forge visp
```

### Step 2: Set Up MegaPose Server

MegaPose runs as a **server** that ViSP calls over a socket. This separation means:
- MegaPose server runs on a GPU machine (Python + PyTorch)
- ViSP tracking runs on the same or different machine (C++/Python)

```bash
# Clone MegaPose
git clone https://github.com/agimus-project/happypose.git
cd happypose
pip install -e .[megapose]

# Download pretrained weights
python -m happypose.toolbox.utils.download --megapose-models

# Start the MegaPose server
python -m happypose.toolbox.server.megapose_server --port 5555
```

### Step 3: Prepare Your Object Model

```
my_object/
├── model.obj      ← 3D mesh (from Blender, FreeCAD, 3D scanner, etc.)
├── model.mtl      ← Material definition
└── texture.jpg    ← Texture (photo of object or solid color)
```

**Critical**: Make sure your `.obj` is in **meters** and the origin is at the object's center.

### Step 4: Run the ViSP Tutorial

Official tutorial: https://visp-doc.inria.fr/doxygen/visp-daily/tutorial-tracking-megapose.html

```bash
# ViSP provides a ready-made tutorial binary
tutorial-megapose-live-single-object-tracking \
  --model-path my_object/ \
  --megapose-server-address 127.0.0.1 \
  --megapose-server-port 5555
```

### Required Hardware

| Component | Minimum | Recommended |
|-----------|---------|-------------|
| GPU | NVIDIA 6GB VRAM | RTX 3080+ |
| Camera | Any RGB (USB webcam) | RealSense D435 (RGB-D) |
| RAM | 16 GB | 32 GB |
| OS | Ubuntu 20.04+ | Ubuntu 22.04 |

In [ ]:
# Simulate what a MegaPose client call looks like
# In real code, ViSP sends this request over a socket to the MegaPose server

# ── Mock MegaPose client API ──────────────────────────────────────────

class MockMegaPoseClient:
    """
    Simulates the MegaPose server interface.
    In real code: connects via socket, sends frames, receives poses.
    """
    def __init__(self, model_path, server_address='127.0.0.1', port=5555):
        self.model_path = model_path
        self.server_address = server_address
        self.port = port
        print(f'[MockClient] Would connect to {server_address}:{port}')
        print(f'[MockClient] Model: {model_path}')

    def estimate_pose(self, rgb_image, detection_bbox, K):
        """
        Send frame + bbox + intrinsics → receive 6D pose.
        Returns: dict with 'R' (3x3), 't' (3,), 'score' (float)
        """
        # In real code: serialize and send over socket
        # Here: return synthetic result
        R = euler_to_R(0.25, 0.4, 0.1)
        t = np.array([0.08, -0.04, 0.42])
        return {'R': R, 't': t, 'score': 0.94}

# ── Usage example ─────────────────────────────────────────────────────

# Camera intrinsics
K = np.array([[600, 0, 320],
              [0, 600, 240],
              [0, 0, 1]], dtype=np.float64)

# Create client (would connect to real MegaPose server)
client = MockMegaPoseClient(
    model_path='../assets/sample_model/',
    server_address='127.0.0.1',
    port=5555
)

# Simulate a detection (normally from YOLO or similar)
detection_bbox = {'x': 230, 'y': 190, 'w': 180, 'h': 160}
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)

# Estimate pose
result = client.estimate_pose(dummy_frame, detection_bbox, K)

print('\n[MegaPose Estimation Result]')
print(f'  Score: {result["score"]:.2f}')
print(f'  Translation: {result["t"].round(4)} m')
print(f'  Rotation matrix:')
for row in result['R']:
    print(f'    {row.round(4)}')

# Convert to rvec for OpenCV
rvec, _ = cv2.Rodrigues(result['R'])
tvec = result['t'].astype(np.float32)
print(f'\n  rvec: {rvec.flatten().round(4)}')
print(f'  Ready for cv2.drawFrameAxes()')

## 6 — MegaPose vs FoundationPose vs FreeZe

| Feature | MegaPose | FoundationPose | FreeZe |
|---------|----------|----------------|--------|
| **Year** | 2022 | 2024 | 2024 |
| **Organization** | INRIA | NVIDIA | Snap Research |
| **Estimation** | Yes (slow init) | Yes | Yes |
| **Tracking** | Via ViSP | Built-in | Not built-in |
| **RGB-D required** | No (RGB only) | Yes | Yes |
| **Training needed** | No (pretrained) | No (pretrained) | No |
| **CAD model needed** | Yes | Yes (or ref imgs) | Yes |
| **BOP rank** | Top 5 | Top 3 | Top 3 |
| **Real-time capable** | Init only (~2s) | Yes (~30FPS) | ~10 FPS |
| **Best for** | RGB-only, no depth | Best accuracy | Zero-shot, no training |

### Practical Decision Guide

```
Have depth camera (RealSense/Kinect)?
  YES → FoundationPose (best accuracy) or FreeZe (zero-shot)
  NO  → MegaPose + ViSP (works with RGB only)

Need to track pose in real-time video?
  YES → MegaPose + ViSP (fast tracking after slow init)
       OR FoundationPose (built-in tracking)
  NO  → Any estimator works

New objects arrive daily with no retraining?
  YES → FreeZe or FoundationPose (both generalize well)
```

In [ ]:
# Visualize what the ViSP model-based tracker shows during operation

def draw_3d_box_edges(img, corners_2d, color=(0, 255, 0), thickness=2):
    """
    Draw 3D bounding box edges on an image.
    corners_2d: (8, 2) array of projected 3D box corners.
    """
    edges = [
        (0,1),(1,2),(2,3),(3,0),   # bottom face
        (4,5),(5,6),(6,7),(7,4),   # top face
        (0,4),(1,5),(2,6),(3,7)    # vertical edges
    ]
    for i, j in edges:
        p1 = tuple(corners_2d[i].astype(int))
        p2 = tuple(corners_2d[j].astype(int))
        cv2.line(img, p1, p2, color, thickness)

def project_box_corners(R, t, box_size, K):
    """Project 3D box corners to 2D."""
    s = box_size / 2
    corners_3d = np.array([
        [-s, -s, -s], [s, -s, -s], [s, s, -s], [-s, s, -s],
        [-s, -s,  s], [s, -s,  s], [s, s,  s], [-s, s,  s]
    ], dtype=np.float32)
    rvec, _ = cv2.Rodrigues(R)
    dist = np.zeros(4)
    corners_2d, _ = cv2.projectPoints(corners_3d, rvec, t.astype(np.float32), K, dist)
    return corners_2d.reshape(-1, 2)

# Simulate 3 tracking frames
K = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]], dtype=np.float64)
box_size = 0.08

poses = [
    (euler_to_R(0.30, 0.40, 0.10), np.array([0.05, -0.03, 0.45], dtype=np.float32)),
    (euler_to_R(0.31, 0.41, 0.10), np.array([0.05, -0.03, 0.44], dtype=np.float32)),
    (euler_to_R(0.32, 0.42, 0.11), np.array([0.06, -0.03, 0.43], dtype=np.float32)),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
frame_labels = ['Frame 0\n(MegaPose init)', 'Frame 1\n(ViSP track)', 'Frame 5\n(ViSP track)']

for ax, (R, t), label in zip(axes, poses, frame_labels):
    # Create synthetic scene
    scene = np.full((480, 640, 3), (35, 35, 45), dtype=np.uint8)
    # Add fake object background
    cv2.rectangle(scene, (260, 190), (380, 310), (90, 110, 130), -1)

    # Project 3D box
    corners_2d = project_box_corners(R, t, box_size, K)
    # Clamp corners to image
    corners_2d[:, 0] = np.clip(corners_2d[:, 0], 0, 639)
    corners_2d[:, 1] = np.clip(corners_2d[:, 1], 0, 479)

    draw_3d_box_edges(scene, corners_2d, color=(0, 255, 0), thickness=2)

    # Draw axes
    rvec, _ = cv2.Rodrigues(R)
    dist = np.zeros(4)
    try:
        cv2.drawFrameAxes(scene, K, dist, rvec, t, box_size * 0.8)
    except Exception:
        pass

    cv2.putText(scene, label.split('\n')[0], (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontweight='bold')
    ax.axis('off')

plt.suptitle('MegaPose + ViSP: Model-Based Tracking Output', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Recap

| Concept | Key Point |
|---------|----------|
| MegaPose | Render-and-compare: encodes renders + scene crop, finds best match |
| CAD model format | `model.obj` + `model.mtl` + `texture.jpg` in one folder |
| Coarse estimator | Matches scene features to rendered viewpoint features |
| Refiner | Iterative Δ[R\|t] correction using render-vs-scene difference |
| ViSP | Classical edge-based tracker (5ms/frame), used after MegaPose init |
| MegaPose server | Runs separately on GPU; ViSP calls it over a socket |
| RGB only | MegaPose works without depth — unlike FoundationPose/FreeZe |
| Best combo | MegaPose (init) + ViSP (track) → slow init, real-time tracking |

In [ ]:
# ============================================================
# EXERCISE 1: Create a box OBJ with a custom size
# ============================================================
# Use the create_box_obj() function to create a 12cm x 8cm x 6cm box.
# Note: The function creates a CUBE. You'll need to modify it
# to support non-uniform dimensions.
# Hint: replace the single 's' variable with sx, sy, sz.

# YOUR CODE HERE

# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def create_box_obj_custom(sx=0.06, sy=0.04, sz=0.03):
#     obj = f"""# Custom box model {sx*2}x{sy*2}x{sz*2} m
# mtllib model.mtl
# v -{sx} -{sy} -{sz}\nv  {sx} -{sy} -{sz}\nv  {sx}  {sy} -{sz}\nv -{sx}  {sy} -{sz}
# v -{sx} -{sy}  {sz}\nv  {sx} -{sy}  {sz}\nv  {sx}  {sy}  {sz}\nv -{sx}  {sy}  {sz}
# vt 0 0\nvt 1 0\nvt 1 1\nvt 0 1
# vn 0 0 -1\nvn 0 0 1\nvn -1 0 0\nvn 1 0 0\nvn 0 -1 0\nvn 0 1 0
# usemtl Material
# f 1/1/1 2/2/1 3/3/1\nf 1/1/1 3/3/1 4/4/1
# f 5/1/2 7/3/2 6/2/2\nf 5/1/2 8/4/2 7/3/2
# """
#     return obj
#
# print(create_box_obj_custom(sx=0.06, sy=0.04, sz=0.03)[:200])

In [ ]:
# ============================================================
# EXERCISE 2: Simulate pose tracking drift over 30 frames
# ============================================================
# Start with the initial pose from MockMegaPoseClient.estimate_pose()
# Apply simulate_visp_track() for 30 frames.
# Plot the Z distance (depth) over time.
# Observation: what happens to depth as the object moves toward camera?

# YOUR CODE HERE
# client = MockMegaPoseClient(...)
# result = client.estimate_pose(...)
# R, t = result['R'], result['t']
# depths = []
# for i in range(30):
#     R, t = simulate_visp_track(R, t, i)
#     depths.append(t[2])
# plt.plot(depths)

# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# client = MockMegaPoseClient('my_object/')
# result = client.estimate_pose(dummy_frame, detection_bbox, K)
# R, t = result['R'], result['t'].copy()
# depths = [t[2]]
# for i in range(1, 30):
#     R, t = simulate_visp_track(R, t, i)
#     depths.append(t[2])
# plt.figure(figsize=(8, 4))
# plt.plot(depths, 'b-o', ms=4)
# plt.xlabel('Frame'); plt.ylabel('Z depth (m)')
# plt.title('Object Z depth over 30 tracked frames')
# plt.grid(True, alpha=0.3); plt.show()
# # Z decreases → object is approaching the camera